# Task 1

Fit a linear regression which describes tree age (Dataset dab.csv). Use numpy.linalg library to solve the equation $X^TX=X^Ty$. Present dataset and the model on a picture.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# loading data and dropping missing values
Data = pd.read_csv('dab.csv')
Data = pd.DataFrame(Data).dropna()
# changing second data type from string to float
Data['srednica2'] = Data['srednica2'].str.replace(',', '.').astype(float) # vectorized operation



In [ ]:
# checking linearity
r, p_value = stats.pearsonr(Data['srednica'], Data['srednica2'])
print(f'R value: {r}\nP value: {p_value}')

We can see strong linear dependence between 'srednica1' and 'srednica2'

In [ ]:
# changing data type 
np_srednica = Data['srednica'].to_numpy().reshape(-1, 1)
# creating matrix with beta intercept coefficient
X = np.hstack([np.ones((np_srednica.shape[0], 1)), np_srednica])
b = Data['wiek'].to_numpy().reshape(-1, 1)
# solving equation
intercept_solve, coef_solve = np.linalg.solve(X.T @ X, X.T @ b)

In [ ]:
x_plot = np_srednica.reshape(-1, )
y_plot = b.reshape(-1, )

plt.plot(x_plot, coef_solve * x_plot + intercept_solve)
plt.scatter(x_plot, y_plot)
plt.title(f'manual coefficent = {coef_solve}')
RSS = sum((x_r - x_e)**2 for x_r, x_e in zip())

# Task 2

Perform the above task using the function $\texttt{LinearRegression()}$ from $\verb+sklearn.linear_model+$.

In [ ]:
# uwaga to bierze za argument df !!! a potem 
X_train, X_test, y_train, y_test = train_test_split(Data[['srednica']], Data['wiek'], test_size=0.2)
model = LinearRegression()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
# comparison 
model = LinearRegression()
model.fit(Data[['srednica']], Data['wiek'])
# new model coef, old model coef, intercent interct
model.coef_, coef_solve, model.intercept_, intercept_solve

# Task 3

Create an artificial dataset ($x_i,y_i$, with $n=1000$) such that $y=x^2+e$, where $e$ comes from the standard normal distribution and $x\in[-1,1]$. 
Next, split the dataset into two parts (for $x<0$ and $x>0$) and train a linear regression $y=\beta_0 + \beta_1x$ on the first part. Please, read coefficients, $R^2$ and draw histogram of residuals.

Calculate the mean squared error (MSE) $\frac{\sum_{i=1}^{n/2}(\hat{y_i}-y_i)^2}{n/2}$ for the values fitted on the traning set($x<0$) and (next) on the test set($x>0$). Please compare this two values.

In [ ]:
# parameters 
n = 1000
seed = 3
scale_par = 1
# creating data set
x = stats.uniform(loc=-1, scale= 2).rvs(n, random_state=seed).reshape(-1, 1)
y = x**2 + stats.norm(scale=scale_par).rvs(n, random_state=seed).reshape(-1, 1)
df = pd.DataFrame(data=np.hstack((x, y)), columns=['X', 'Y'])

# splitting sets 
filt_pos = df['X'] > 0
X_train, X_test, y_train, y_test = df[filt_pos][['X']], df[~filt_pos][['X']], df[filt_pos]['Y'], df[~filt_pos]['Y']

In [ ]:
# learning model on data X > 0 
model = LinearRegression()
model.fit(X_train, y_train)
# getting scores (on all data)
score = model.score(df[['X']], df['Y'])  
coef, intercept = model.coef_.item(), model.intercept_
# residuals differences between true value, estimated value
df['Residuals'] = df[['Y']] - model.predict(df[['X']]).reshape(-1, 1)
# printing values
print(f'Score: {score}\nCoefficent: {coef}\nIntercept: {intercept}')
plt.hist(df[['Residuals']], bins=20, edgecolor='black')
# TODO ~ does we can estimate error based on residual value

In [ ]:
# calculating MSE 
MSE_pos = mean_squared_error(y_train, model.predict(X_train))
MSE_neg = mean_squared_error(y_test, model.predict(X_test))
print(f'positive: {MSE_pos}\nnegative: {MSE_neg}')

# Task 4

Repeat Task 3 for some better linear regression model (you can use a data transformation $\texttt{PolynomialFeatures}$). 

Again read $R^2$ and MSE for fitted values on both parts of data.



In [ ]:
from sklearn.preprocessing import PolynomialFeatures
# include bias false ~ important: it adds another columns with ones, linear regression do it for ous so somewhere we need to decline it 
poly = PolynomialFeatures(degree=2, include_bias=False)  
# changing data to fit model
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
# training model
model_poly = LinearRegression()
model_poly.fit(X_train_poly, y_train)
y_poly_predict_pos = model_poly.predict(X_train_poly)
y_poly_predict_neg = model_poly.predict(X_test_poly)
# calculating scores
poly_score = model_poly.score(X_test_poly, y_test)
# poly_score = r2_score(y_test, y_poly_predict_neg)
MSE_pol_pos = mean_squared_error(y_train, y_poly_predict_pos)
MSE_pol_neg = mean_squared_error(y_test, y_poly_predict_neg)
print(f'model score: {poly_score}\npositive: {MSE_pol_pos}\nnegative: {MSE_pol_neg}')

In [ ]:
domain = np.linspace(-1, 1, 100).reshape(-1, 1)
poly_domain = poly.transform(domain)
plt.figure(figsize=(10, 6))
# Original data points
plt.scatter(x, y, s=10, alpha=0.3, color='gray', label='Data points')
# Linear regression line
plt.plot(domain, model.predict(domain), color='red', linewidth=2, label='Linear Regression', linestyle='--')
# Polynomial regression curve
plt.plot(domain, model_poly.predict(poly_domain), color='blue', linewidth=2, label='Polynomial Regression (deg=2)', linestyle='-')

# Grid, labels and title
plt.title("Linear vs Polynomial Regression", fontsize=14)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()


# Task 5

In [ ]:
from sklearn.model_selection import cross_val_score

Perform cross-validation  $\texttt{cross_val_score}$ for models from Task 1,2

In [ ]:
model = LinearRegression()
scores = cross_val_score(model, Data[['srednica']], Data['wiek'], cv=5, )
scores